# Credit Risk Intelligence — Fast Full 307,511-Row Pipeline (Google Colab)

Runs the complete unified platform on the full Home Credit dataset (`307,511` training rows + `48,744` test rows + all child history tables) using Colab's fast local SSD (`/content/spark_work`) to avoid slow Google Drive FUSE I/O:
1. Provision & verify OpenJDK 17/21 and set a valid `JAVA_HOME`.
2. Mount Google Drive and locate your repository + `_data/` folder.
3. Install Python & Spark/Delta dependencies (`pyspark==3.5.3`, `delta-spark==3.2.1`, `lightgbm`, `shap`).
4. Run quick Spark + Delta Lake smoke verification.
5. Stage `_data/` to local SSD and run the full unified pipeline (Bronze -> Silver -> Gold + Streaming + Real-Time Model + Gold LightGBM/LR/Cost-Opt/SHAP).
6. Download `credit_risk_full_artifacts.zip` (also saved directly into your Drive repo folder) and extract it locally.

In [ ]:
#@title 1. Provision Java (JDK 17 preferred, fallback to pre-installed JDK 21) & verify JAVA_HOME
!rm -f /etc/apt/sources.list.d/*cran* /etc/apt/sources.list.d/*r2u* 2>/dev/null || true
!apt-get update -qq || true
!apt-get install -y -qq openjdk-17-jdk-headless > /dev/null || true

import os, shutil
from pathlib import Path

java_home = None
for candidate in (
    '/usr/lib/jvm/java-17-openjdk-amd64',
    '/usr/lib/jvm/java-17-openjdk',
    '/usr/lib/jvm/java-21-openjdk-amd64',
    '/usr/lib/jvm/default-java',
):
    if (Path(candidate) / 'bin' / 'java').exists():
        java_home = candidate
        break
if not java_home and shutil.which('java'):
    java_home = str(Path(shutil.which('java')).resolve().parents[1])

assert java_home and (Path(java_home) / 'bin' / 'java').exists(), 'No working Java runtime found!'
os.environ['JAVA_HOME'] = java_home
os.environ['PATH'] = f"{java_home}/bin:" + os.environ.get('PATH', '')
print('Verified JAVA_HOME =', os.environ['JAVA_HOME'])
!"$JAVA_HOME/bin/java" -version
!python --version

In [ ]:
#@title 2. Mount Google Drive and locate synced repository + _data/
from google.colab import drive
from pathlib import Path
import os

drive.mount('/content/drive')

CANDIDATE_DIRS = [
    '/content/drive/MyDrive/_hust/IT5427 - Big Data',
    '/content/drive/MyDrive/_hust/IT5427 - Big Data/it5427-credit_risk_bigdata',
    '/content/drive/MyDrive/it5427-credit_risk_bigdata',
]
REPO_DIR = next(
    (p for p in CANDIDATE_DIRS if (Path(p) / 'run_track_a.py').exists() and (Path(p) / '_data' / 'application_train.csv').exists()),
    None,
)
assert REPO_DIR is not None, f'Could not find repo + _data/application_train.csv in: {CANDIDATE_DIRS}'
os.chdir(REPO_DIR)
print('Working repo:', os.getcwd())
!ls -lh _data/

In [ ]:
#@title 3. Install Python dependencies (ignore benign dataproc-spark-connect warning)
!pip install -q -r requirements.txt -r apache-spark/requirements.txt
!python -c "import pyspark, delta, lightgbm, shap; print('Deps OK | PySpark:', pyspark.__version__)"

In [ ]:
#@title 4. Quick Spark & Delta Lake connectivity check (~10s)
%cd apache-spark
!CRB_TEST_DELTA=1 python -m pytest tests/test_06_delta_smoke.py -v
# Optional full 41-test Spark suite (adds a few minutes before pipeline start):
# !CRB_TEST_DELTA=1 python -m pytest tests/ -v
%cd ..

In [ ]:
#@title 5. Run Unified Full 307,511-Row Pipeline (staged on local SSD for 5-10x speedup)
# Remove --fast if you also want to re-run pandas EDA + Spark column-level raw profiling.
!python scripts/run_colab_full_pipeline.py \
  --raw-dir _data \
  --work-dir /content/spark_work \
  --stage-raw \
  --fast \
  --driver-memory auto \
  --shuffle-partitions 32 \
  --verify-delta \
  --bundle credit_risk_full_artifacts.zip

In [ ]:
#@title 6. Download generated artifact bundle (also saved inside Drive REPO_DIR)
from google.colab import files
files.download('credit_risk_full_artifacts.zip')